# Self-Attention with Trainable Q, K, V Weights

Real self-attention uses three learned weight matrices (W_query, W_key, W_value) to project
each token embedding into separate query, key, and value spaces.

- **Query**: what this token is looking for
- **Key**: what this token advertises to others
- **Value**: what this token actually contributes to the output

Scores are scaled by sqrt(d_k) to prevent dot products from growing too large in high dimensions.

In [ ]:
import torch
import torch.nn as nn

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

d_in = inputs.shape[1]  # 3 — input embedding dimension
d_out = 2               # output dimension (can differ from d_in)

## Manual weight matrices (for illustration)

Normally these are learned during training. Here we initialise them manually to see the mechanics.

In [ ]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

x_2 = inputs[1]  # "journey" token

query_2 = x_2 @ W_query
key_2   = x_2 @ W_key
value_2 = x_2 @ W_value
print("query_2:", query_2)

## Compute attention scores and context vector for token 2

In [ ]:
keys    = inputs @ W_key
values  = inputs @ W_value

attn_scores_2 = query_2 @ keys.T
print("Attention scores:", attn_scores_2)

d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print("Attention weights:", attn_weights_2)

context_vec_2 = attn_weights_2 @ values
print("Context vector:", context_vec_2)

## SelfAttention_v1 — nn.Parameter

Wraps the above logic into a PyTorch module. W_query/W_key/W_value are `nn.Parameter`
so they get updated during backprop.

In [ ]:
class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys    = x @ self.W_key
        queries = x @ self.W_query
        values  = x @ self.W_value

        attn_scores  = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        return attn_weights @ values


torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

## SelfAttention_v2 — nn.Linear

`nn.Linear` is preferred over raw `nn.Parameter` — it uses better weight initialisation
and supports an optional bias term.

In [ ]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)

        attn_scores  = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        return attn_weights @ values


torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))